# DSFB-Debug — Reviewer Reproducibility Notebook

**Deterministic detector-field semiotics with routed forensic witness-field fusion**

This notebook runs the full DSFB-Debug crate from scratch on a fresh Colab runtime, against twelve real-bytes vendored fixtures spanning nine distinct upstream public datasets, and produces:

- 126 figures (10 per fixture × 12 fixtures + 3 architecture/infrastructure + 3 cross-fixture summary)
- 12 verbatim JSON metric blocks (one per fixture)
- A single multi-page PDF report (`report.pdf`) assembled via `matplotlib.PdfPages`
- A zipped artefact bundle for one-click download

Output folder is timestamped: `output-dsfb-debug/dsfb-debug-{ISO-datetime}/`. Reruns never overwrite each other.

---

## Prior Art Notice (35 U.S.C. § 102)

This notebook constitutes prior art deposited via Zenodo + GitHub at the run-id timestamp encoded in the parent folder name. All rights reserved to prior-art date of public deposit.

## IP Notice

The theoretical framework, formal constructions, and supervisory methods described herein constitute proprietary Background IP of **Invariant Forge LLC** (Delaware LLC No. 10529072), with prior art established by this publication and earlier Zenodo DOI publications by the same author. Commercial deployment requires a separate written license. Reference implementations are released under Apache 2.0.

**Licensing:** `licensing@invariantforge.net`

## Authoring discipline

- All numerical values shown are verbatim test stdout from the engine. **No synthetic data, ever.**
- Every detector is a deterministic statistical or structural function with a literature citation. **No neural network, no learned model, no training data.**
- Every motif is a hand-curated rule anchored to **IEEE 24765 / Avizienis-Laprie-Randell** vocabulary.
- The reference Rust implementation is `#![no_std]` + zero-allocation + `#![forbid(unsafe_code)]` with zero runtime Cargo dependencies in the core; it compiles for ARM Cortex-M / RISC-V embedded targets.
- Theorem 9 (deterministic replay) holds across every fixture; this notebook re-fires `verify_deterministic_replay` per evaluation.

---

## Math summary — the engine in one page

**Residual signature** at window $k$, signal $s$:

$$\sigma(k) = (\|r(k)\|,\ \dot{r}(k),\ \ddot{r}(k))$$

where $r(k) = x(k) - \hat{x}(k)$ is the residual, $\dot{r}$ is finite-difference drift over a fixed window $W$, $\ddot{r}$ is slew (first difference of drift).

**Admissibility envelope:** $\mathcal{E}(k) = \{r : \|r\| \leq \rho(k)\}$ where $\rho$ is operator-defined (SLO/SLA target or healthy-window baseline).

**Grammar state** $G(k,s) \in \{\text{Admissible},\ \text{Boundary},\ \text{Violation}\}$ is a deterministic function of $\sigma(k,s)$ and $\rho$, with hysteresis confirmation (n_confirm windows).

**Detector ensemble:** 205 deterministic detectors organised across 27 mathematical axes (Tiers A–U + EXTRA + V/X/Y/Z/AA). Each detector is a pure function of the residual matrix; outputs the per-(window, signal) firing pattern.

**Routed Evidence Principle:** for each motif $m$ in the bank, an affinity bitmask $\alpha(m)$ over the 27 tiers selects which detectors' evidence enters the motif's score. The motif-conditional consensus at cell $(w,s)$ is

$$c_m(w,s) = \mathrm{popcount}\bigl(\alpha(m) \wedge \phi(w,s)\bigr)$$

where $\phi(w,s)$ is the per-cell tier-fired bitmask.

**Episode aggregation** collapses contiguous non-Admissible windows into typed structural episodes (`DebugEpisode`), each with a `StructuralSignature` (peak slew, duration, contributing signals, dominant drift direction).

**9-axis bank-aware fusion:** provenance gate, margin gate, tier-affinity scoring, zero-tier-firing filter, adaptive margin gate, confuser-boundary adjudication, structural disambiguator boost, tier-level primary witness gate, per-detector named witness gate. Each axis is a single `FusionConfig` flag.

**Operator evidence packet** per typed episode: top motif, runner-up, declared confuser, three margins, tier consensus factor, disambiguator boost, witness-gate flags, root-cause attribution.

---

## Code tour — public API

```rust
// Engine creation under paper-lock parameters
let engine = DsfbDebugEngine::<32, 64>::paper_lock()?;

// Real-data evaluation against a vendored fixture
let eval = evaluate_real_dataset(&engine, &MANIFEST_TADBENCH_F11, F11_BYTES)?;

// Multi-detector fusion (9-axis bank-aware)
let metrics = run_fusion_evaluation(
    &engine, &data, num_signals, num_windows,
    healthy_window_end, &fault_labels,
    &FusionConfig::ALL_DEFAULT, fixture_name,
)?;

// Per-episode evidence packet
let bank = HeuristicsBank::<64>::with_canonical_motifs();
let confidence = bank.match_episode_with_consensus(&episode, /* ... */);
```

All entry points accept `&[T]` only; the type system enforces non-intrusion at compile time.

## Step 1 — Setup

Install rustup (1.85+ for the `demo` feature), clone the crate, and build the demo binary.

In [ ]:
# Install rustup (~30 seconds on Colab)
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain 1.85.1 -q
import os
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
!rustc --version

In [ ]:
# Clone the dsfb repository (or use a local path if running outside Colab)
import os
if not os.path.isdir('dsfb'):
    !git clone https://github.com/infinityabundance/dsfb.git
%cd dsfb/crates/dsfb-debug
!ls data/fixtures/ | wc -l    # expect 12 fixtures

In [ ]:
# Pin time + time-core for MSRV-compatible build, then build the demo binary.
!cargo update -p time --precise 0.3.41 2>&1 | tail -5
!cargo update -p time-core --precise 0.1.4 2>&1 | tail -5
!cargo build --release --features demo --bin dsfb-debug-demo 2>&1 | tail -3

# Install Python figure-quality packages (publication style + perceptually-uniform colormaps)
!pip install --quiet --break-system-packages scienceplots cmcrameri 2>&1 | tail -3

## Step 2 — Run the demo binary

Produces 126 PNG figures + 12 JSON metric blocks + Markdown report + zip.

In [ ]:
import subprocess
result = subprocess.run(['cargo', 'run', '--release', '--features', 'demo', '--bin', 'dsfb-debug-demo'],
                       capture_output=True, text=True)
print(result.stdout[-2000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

# Locate the run folder + zip path
import glob
run_dirs = sorted(glob.glob('output-dsfb-debug/dsfb-debug-*'))
run_dir = [d for d in run_dirs if not d.endswith('.zip')][-1]
zip_path = run_dir + '.zip'
print('Run folder:', run_dir)
print('Zip:', zip_path)

# Run the Python publication-quality figure pipeline
print('\n[python] Rendering publication-quality figures via matplotlib…')
result2 = subprocess.run(['python3', '-m', 'tools.figures.render', run_dir],
                         capture_output=True, text=True)
print(result2.stdout[-1500:])
if result2.returncode != 0:
    print('STDERR:', result2.stderr[-1000:])
figs_py_dir = os.path.join(run_dir, 'figures-py')
print('Python-rendered figures:', figs_py_dir)

## Step 3 — Display architecture + cross-fixture figures inline

In [ ]:
from IPython.display import Image, display, Markdown
import os

# Display architecture / infrastructure figures (publication-quality, matplotlib-rendered)
for label, fname in [
    ('Figure A1 — How DSFB-Debug differs from learned anomaly detection', '00_architecture/01_ml_vs_dsfb.png'),
    ('Figure A2 — Detector ensemble (205 detectors / 27 axes / 8 mathematical families)', '00_architecture/02_tier_breakdown.png'),
    ('Figure A3 — 32-motif × 27-axis affinity matrix (Routed Evidence Principle)', '00_architecture/03_motif_affinity.png'),
    ('Figure C1 — F-11 fusion sweep: clean-window FP rate vs min_consensus N', 'cross_fixture/01_fusion_sweep.png'),
    ('Figure C2 — Cross-fixture RSCR forest plot (12 fixtures)', 'cross_fixture/02_rscr_forest.png'),
    ('Figure C3 — Cross-fixture per-tier firing pattern', 'cross_fixture/03_tier_firing.png'),
    ('Figure O1 — F-11 forensic evidence packets (3-card panel)', 'operator/01_evidence_cards.png'),
    ('Figure O2 — F-11 episode 0 confuser-pair adjudication (Phase 5.6 gate)', 'operator/02_confuser_adjudication.png'),
    ('Figure O3 — Anti-Hallucination Ladder progression on F-11', 'operator/03_anti_hallucination_ladder.png'),
    ('Figure S1 — Trace Event Collapse on F-11', 'special/01_trace_event_collapse.png'),
    ('Figure S2 — Theorem 9 deterministic-replay verification', 'special/02_theorem9_verification.png'),
]:
    p = os.path.join(figs_py_dir, fname)
    if os.path.isfile(p):
        display(Markdown(f'### {label}'))
        display(Image(p))


## Step 4 — Per-fixture figures + verbatim JSON metrics

12 fixtures × 10 figures = 120 figures inline, plus the JSON metric block per fixture.

In [ ]:
import json, glob
fixture_dirs = sorted([d for d in glob.glob(os.path.join(figs_py_dir, '[0-9]*'))
                        if os.path.isdir(d)])

for d in fixture_dirs:
    label = os.path.basename(d)
    display(Markdown(f'## {label}'))
    json_path = os.path.join(run_dir, 'results', f'{label}.json')
    if os.path.isfile(json_path):
        with open(json_path) as f:
            data = json.load(f)
        # Just show the headline metrics (full JSON is multi-MB)
        head = {
            'manifest_name': data.get('manifest_name'),
            'episode_count': data.get('episode_count'),
            'metrics': data.get('metrics'),
            'fusion': data.get('fusion'),
        }
        display(Markdown('**Verbatim metric block:**'))
        display(Markdown('```json\n' + json.dumps(head, indent=2) + '\n```'))
    for fn in ['01_residual_smallmult.png', '02_summary_card.png', '03_episode_evidence.png']:
        p = os.path.join(d, fn)
        if os.path.isfile(p):
            display(Image(p))


## Step 5 — Assemble PDF report (via matplotlib.PdfPages)

Combines all 126 PNG figures into a single PDF with one figure per page, captioned.

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob, os

pdf_path = os.path.join(run_dir, 'report.pdf')
all_figs = []
all_figs += sorted(glob.glob(os.path.join(figs_py_dir, '00_architecture', '*.png')))
all_figs += sorted(glob.glob(os.path.join(figs_py_dir, 'cross_fixture', '*.png')))
all_figs += sorted(glob.glob(os.path.join(figs_py_dir, 'operator', '*.png')))
all_figs += sorted(glob.glob(os.path.join(figs_py_dir, 'special', '*.png')))
for d in sorted([d for d in glob.glob(os.path.join(figs_py_dir, '[0-9]*'))
                  if os.path.isdir(d)]):
    all_figs += sorted(glob.glob(os.path.join(d, '*.png')))

with PdfPages(pdf_path) as pdf:
    # Cover page
    fig = plt.figure(figsize=(8.27, 11.69))   # A4 portrait
    fig.text(0.05, 0.92, 'DSFB-Debug — Demo Report', fontsize=22, weight='bold')
    fig.text(0.05, 0.87, 'Deterministic detector-field semiotics with routed forensic witness-field fusion', fontsize=11)
    fig.text(0.05, 0.83, 'Twelve real-bytes vendored fixtures across nine distinct upstream public datasets.', fontsize=10)
    fig.text(0.05, 0.75, 'Prior Art Notice (35 U.S.C. §102):', fontsize=10, weight='bold')
    fig.text(0.05, 0.72, 'Deposited via Zenodo + GitHub at the run-id timestamp.', fontsize=9)
    fig.text(0.05, 0.66, 'IP Notice:', fontsize=10, weight='bold')
    fig.text(0.05, 0.59, 'Background IP of Invariant Forge LLC. Reference implementation Apache-2.0;\n'
                          'commercial deployment requires written license.\n'
                          'Contact: licensing@invariantforge.net', fontsize=9)
    fig.text(0.05, 0.50, 'Authoring discipline:', fontsize=10, weight='bold')
    fig.text(0.05, 0.43, 'All numbers are verbatim test stdout from the engine. No synthetic data.\n'
                          'Theorem 9 (deterministic replay) holds across every fixture.\n'
                          'ML-free, no_std, zero runtime Cargo dependencies in the core, edge-deployable.', fontsize=9)
    plt.axis('off')
    pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    for fp in all_figs:
        fig = plt.figure(figsize=(8.27, 11.69))   # A4 portrait
        img = mpimg.imread(fp)
        ax = fig.add_axes([0.05, 0.10, 0.90, 0.80])
        ax.imshow(img)
        ax.axis('off')
        rel = os.path.relpath(fp, run_dir)
        fig.text(0.05, 0.96, rel, fontsize=8, family='monospace')
        pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

print(f'OK — wrote {pdf_path}, {len(all_figs)+1} pages')


## Step 6 — Re-zip with PDF included, then download

In [ ]:
import zipfile, os
rezip = os.path.join('output-dsfb-debug', os.path.basename(run_dir) + '-with-pdf.zip')
with zipfile.ZipFile(rezip, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(run_dir):
        for f in files:
            p = os.path.join(root, f)
            arc = os.path.relpath(p, os.path.dirname(run_dir))
            z.write(p, arc)
print(f'Zip ready: {rezip}')

## Step 7 — Calibration & Sensitivity walk-through (Phase η.7, Session 18)

The next two cells run the Session-18 sensitivity-sweep + per-axis-ablation
audits over the F-11 fixture and render the response curves. These are
the operator-side calibration tools: an on-call engineer pointing
DSFB-Debug at their own trace data can re-run these cells with their
own fixture path to see how each parameter and each fusion axis behaves
on their data.

Source artefacts (already populated by `cargo test --features "std paper-lock"`):
- `docs/audit/sensitivity_sweep.md` — 5 parameters × 5 values response table
- `docs/audit/axis_ablation.md` — 9-axis ladder ablation impact
- `docs/audit/bootstrap_ci.md` — 95% CI on cross-fixture aggregates
- `docs/audit/kfold_cv.md` — K-fold CV (K=4) per-fold + cross-fold aggregate
- `docs/audit/detector_subset_opt.md` — minimal-sufficient-subset trajectory
- `docs/benchmarks.md` — wall-clock + ns/cell timing


In [ ]:
# Phase η.7a — Render sensitivity-sweep response curves.
#
# Reads `docs/audit/sensitivity_sweep.md`, parses the 5 per-parameter
# tables, plots typed-confirmed and FP rate vs each parameter on a
# 2×3 grid. Operator can correlate which parameters matter most
# for their site.
import re
from pathlib import Path
import matplotlib.pyplot as plt

audit_path = Path("crates/dsfb-debug/docs/audit/sensitivity_sweep.md")
if not audit_path.exists():
    audit_path = Path("docs/audit/sensitivity_sweep.md")

if audit_path.exists():
    text = audit_path.read_text()
    # Parse each per-parameter table.
    sections = re.split(r'\n## ([A-Za-z0-9_ ()-]+)\n', text)
    fig, axes = plt.subplots(2, 3, figsize=(13, 7), constrained_layout=True)
    axes = axes.flatten()
    plotted = 0
    for i in range(1, len(sections), 2):
        param = sections[i].strip()
        body = sections[i+1] if i+1 < len(sections) else ""
        rows = re.findall(r'\| ([0-9.]+) \| ([0-9.]+) \| ([0-9.]+) \| ([0-9.]+) \| ([0-9]+) \|', body)
        if not rows or plotted >= 6:
            continue
        xs = [float(r[0]) for r in rows]
        rscr = [float(r[1]) for r in rows]
        fp = [float(r[2]) for r in rows]
        recall = [float(r[3]) for r in rows]
        typed = [int(r[4]) for r in rows]
        ax = axes[plotted]
        ax2 = ax.twinx()
        ax.plot(xs, fp, 'o-', label='FP rate', color='#9A031E')
        ax.plot(xs, recall, 's-', label='Recall', color='#0F4C5C')
        ax2.plot(xs, typed, 'd--', label='Typed', color='#E36414')
        ax.set_xlabel(param)
        ax.set_ylabel('FP rate / Recall')
        ax2.set_ylabel('Typed-confirmed')
        ax.set_title(param, fontsize=10)
        ax.legend(loc='upper left', fontsize=8)
        ax2.legend(loc='upper right', fontsize=8)
        ax.grid(alpha=0.3)
        plotted += 1
    for k in range(plotted, 6):
        axes[k].axis('off')
    fig.suptitle('Phase η.3 — Sensitivity sweep response curves\n(verbatim from cargo test, Session 18)', fontweight='bold')
    plt.show()
else:
    print(f"Sensitivity audit not found at {audit_path}; run `cargo test --features \"std paper-lock\" --test sensitivity_sweep -- --nocapture` first.")


In [ ]:
# Phase η.7b — Render per-axis ablation waterfall + bootstrap CI summary.
#
# Reads `docs/audit/axis_ablation.md` + `docs/audit/bootstrap_ci.md` and
# plots: (a) per-axis Δtyped waterfall, (b) bootstrap CI bars on the four
# cross-fixture metrics. Operator sees which fusion axes carry empirical
# load AND the honest uncertainty band on the cross-fixture aggregate.
import re
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

ablation_path = Path("crates/dsfb-debug/docs/audit/axis_ablation.md")
if not ablation_path.exists():
    ablation_path = Path("docs/audit/axis_ablation.md")
boot_path = Path("crates/dsfb-debug/docs/audit/bootstrap_ci.md")
if not boot_path.exists():
    boot_path = Path("docs/audit/bootstrap_ci.md")

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

# (a) Per-axis Δtyped waterfall.
ax = axes[0]
if ablation_path.exists():
    text = ablation_path.read_text()
    rows = re.findall(r'\| (\d+\. [^|]+?) \| `([^`]+)` \| ([0-9.]+) \| ([0-9.]+) \| ([0-9.]+) \| (\d+) \| ([+-]?\d+) \|', text)
    if rows:
        labels = [r[0].strip() for r in rows]
        deltas = [int(r[6]) for r in rows]
        colors = ['#2A9D8F' if d == 0 else ('#9A031E' if d > 0 else '#E36414') for d in deltas]
        ax.barh(range(len(labels)), deltas, color=colors)
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels, fontsize=9)
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_xlabel('Δ typed-confirmed (axis disabled vs baseline)')
        ax.set_title('Phase η.4 — Per-axis ablation impact\n(green = no impact / red = adds typed eps when off / orange = removes when off)', fontsize=10)
        ax.grid(axis='x', alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'Axis ablation: re-run `cargo test --test axis_ablation`', ha='center')
else:
    ax.text(0.5, 0.5, 'Axis ablation audit not found', ha='center')

# (b) Bootstrap CI bars.
ax = axes[1]
if boot_path.exists():
    text = boot_path.read_text()
    rows = re.findall(r'\| ([A-Za-z][^|]+?) \| ([0-9.]+) \| ([0-9.]+) \| ([0-9.]+) \| ([0-9.]+) \|', text)
    if rows:
        labels = [r[0].strip() for r in rows]
        points = [float(r[1]) for r in rows]
        lowers = [float(r[2]) for r in rows]
        uppers = [float(r[3]) for r in rows]
        ys = np.arange(len(labels))
        for i, (p, lo, hi) in enumerate(zip(points, lowers, uppers)):
            # Plot CI bar normalized to point estimate (so bars share a scale).
            if abs(p) > 1e-9:
                ax.barh(i, hi - lo, left=lo, color='#7C7E80', alpha=0.4, height=0.6)
                ax.scatter([p], [i], color='#9A031E', zorder=5, s=80, marker='|')
        ax.set_yticks(ys)
        ax.set_yticklabels(labels, fontsize=9)
        ax.set_xlabel('Bootstrap value (point estimate marked with |)')
        ax.set_title('Phase η.1 — Bootstrap 95% CI\n(grey bar = CI, red mark = point estimate)', fontsize=10)
        ax.grid(axis='x', alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'Bootstrap CI: re-run LO-CV', ha='center')
else:
    ax.text(0.5, 0.5, 'Bootstrap CI audit not found', ha='center')

plt.show()
print("Operator-side calibration audits rendered. The audit ledgers")
print("under docs/audit/ are the source-of-truth for re-running on")
print("site-specific fixtures: edit data/fixtures/ + re-run cargo test.")


In [ ]:
# Download the zip (Colab only)
try:
    from google.colab import files
    files.download(rezip)
except ImportError:
    print(f'Not in Colab. Zip is at: {os.path.abspath(rezip)}')